## tl;dr
Deux sessions : quatre tours A et trois tours B. Résultats exploratoires, sans refit. Environ 0,260 L économisé sur 100–5800 m pour +0,56 s au tour par rapport aux cinq push antérieurs.

## Context & Methods
Compagnon de scripts/analyze_sebring_lico_sessions.py; les sources natives et leurs SHA-256 sont dans manifest.json. Fenêtres figées et références séparées.

### Key Assumptions
Conditions constantes selon le pilote; pas de correction causale de masse. Première tentative : erreurs non localisées. T1 push4 porte un signal d'impact. Les 49 passages ne sont pas 49 observations indépendantes.


## Data
Exécuter depuis la racine LICOR ou le dossier notebooks. Les CSV proviennent de l'analyse native finale; aucune modification des données.


In [ ]:
from pathlib import Path
import json
import hashlib
import polars as pl
root = Path.cwd()
if not (root / 'src/licor').exists():
    root = root.parent
folder = root / 'data/processed/experimental/sebring_lmp2_transfer_2026_09/combined_lico_analysis_final'
manifest = json.loads((folder / 'manifest.json').read_text())
for name, digest in manifest['artifact_hashes'].items():
    assert hashlib.sha256((folder / name).read_bytes()).hexdigest() == digest
laps = pl.read_csv(folder / 'lap_summary.csv').filter(pl.col('complete'))
passes = pl.read_csv(folder / 'zone_passes.csv')
scores = pl.read_csv(folder / 'zone_scores.csv')
cues = pl.read_csv(folder / 'cue_events.csv')
assert laps.height == 14 and laps['quality_ok'].all()
assert cues.height == 49 and (cues['trigger_status'] == 'fired_on_time').all()
assert scores.height == 98  # Two references, not 98 independent observations.
assert not passes.select('run_id', 'lap_number', 'zone_id').is_duplicated().any()
print('Hashes and 14 laps / 49 cues / 98 contrasts verified')


## Results
La somme des fenêtres contiguës doit reproduire exactement le bilan 100–5800 m de chaque tour; les médianes de zones ne doivent pas être additionnées pour reconstruire une médiane de tour.


In [ ]:
summed = passes.group_by('run_id', 'lap_number').agg(pl.col('elapsed_time_s').sum(), pl.col('fuel_used_l').sum())
check = summed.join(laps, on=['run_id', 'lap_number'], suffix='_lap')
for metric in ['elapsed_time_s', 'fuel_used_l']:
    assert check.select((pl.col(metric) - pl.col(metric + '_lap')).abs().max()).item() < 1e-9
lico = laps.filter(pl.col('role').is_in(['A', 'B']))
for cohort in ['prior_push', 'first_attempt']:
    push = laps.filter((pl.col('cohort') == cohort) & (pl.col('role') == 'push'))
    print(cohort, {'fuel_saved_l': push['fuel_used_l'].median() - lico['fuel_used_l'].median(), 'official_time_lost_s': lico['official_time_s'].median() - push['official_time_s'].median()})


In [ ]:
heldout = scores.filter((pl.col('baseline') == 'prior_push_5') & (pl.col('cohort') == 'first_attempt') & pl.col('lap_number').is_in([5, 6]))
assert heldout.height == 14
print('Held-out laps 5/6, driver quality unresolved:', heldout.select((pl.col('fuel_saved_l') - pl.col('predicted_fuel_saved_l')).abs().mean().alias('fuel_mae_l'), (pl.col('time_lost_s') - pl.col('predicted_time_lost_s')).abs().mean().alias('time_mae_s')).to_dicts())


## Takeaways
Aucun refit. L'économie de carburant est encourageante; le coût temporel et le classement restent sensibles à la conduite et aux références. Qualifier les phases et les reports T1/T3 et T17 avant ML. Pas de nouvelle run générale justifiée uniquement par le tour B manquant.

Validation : cellules Python exécutées de haut en bas dans la venv; nbformat/nbclient/ipykernel ne sont pas installés, donc l'exécution par noyau Jupyter n'a pas été vérifiée.
